**Install and Import Dependencies**

In [ ]:
%pip install mediapipe==0.10.5 opencv-python

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

# New API style
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe import solutions

# For drawing utilities (still available here)
mp_drawing = solutions.drawing_utils
mp_pose = solutions.pose

In [ ]:
# VIDEO FEED
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    cv2.imshow('Pose-detection Feed', frame)
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
        
cap.release()
cv2.destroyAllWindows()

**1. Make Detections**

In [ ]:
cap = cv2.VideoCapture(0)
## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
      
        # Make detection
        results = pose.process(image)
    
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2), 
                                mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2) 
                                 )               
        
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [ ]:
mp_drawing.DrawingSpec

2. Determining Joints

<img src="https://i.imgur.com/3j8BPdc.png" style="height: 300px">

In [ ]:
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic


# Drawing specs
pose_landmark_spec = mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2)
pose_connection_spec = mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)

hand_landmark_spec = mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=3)
hand_connection_spec = mp_drawing.DrawingSpec(color=(0, 128, 0), thickness=2, circle_radius=2)

face_landmark_spec = mp_drawing.DrawingSpec(color=(255, 224, 189), thickness=1, circle_radius=1)
face_connection_spec = mp_drawing.DrawingSpec(color=(200, 180, 150), thickness=1, circle_radius=1)

cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    model_complexity=1,              # 0=lite, 1=full, 2=heavy
    smooth_landmarks=True,           # Reduces landmark jitter
    enable_segmentation=False,
    refine_face_landmarks=True,      # Adds iris landmarks for precise eye/iris tracking
) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Flip for natural mirror view
        frame = cv2.flip(frame, 1)

        # Recolor to RGB for MediaPipe
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Run holistic detection (pose + face + hands in one shot)
        results = holistic.process(image)

        # Recolor back to BGR for OpenCV rendering
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # ── 1. FACE MESH (468 landmarks, fits tightly on face geometry) ──────
        if results.face_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.face_landmarks,
                mp_holistic.FACEMESH_CONTOURS,       # Draws face outline + features
                landmark_drawing_spec=face_landmark_spec,
                connection_drawing_spec=face_connection_spec,
            )
            # Optional: draw tesselation (full mesh grid) for a detailed look
            # mp_drawing.draw_landmarks(
            #     image,
            #     results.face_landmarks,
            #     mp_holistic.FACEMESH_TESSELATION,
            #     landmark_drawing_spec=None,
            #     connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style(),
            # )

        # ── 2. RIGHT HAND (21 landmarks per hand) ────────────────────────────
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.right_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
                landmark_drawing_spec=hand_landmark_spec,
                connection_drawing_spec=hand_connection_spec,
            )

        # ── 3. LEFT HAND ──────────────────────────────────────────────────────
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.left_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
                landmark_drawing_spec=hand_landmark_spec,
                connection_drawing_spec=hand_connection_spec,
            )

        # ── 4. FULL BODY POSE (33 landmarks) ─────────────────────────────────
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_holistic.POSE_CONNECTIONS,
                landmark_drawing_spec=pose_landmark_spec,
                connection_drawing_spec=pose_connection_spec,
            )

        # ── 5. Extract & print landmarks (optional) ───────────────────────────
        try:
            pose_lm = results.pose_landmarks.landmark
            # Example: print nose position
            nose = pose_lm[mp_holistic.PoseLandmark.NOSE]
            print(f"Nose → x:{nose.x:.2f}  y:{nose.y:.2f}  visibility:{nose.visibility:.2f}")
        except AttributeError:
            pass

        # ── Display ───────────────────────────────────────────────────────────
        cv2.putText(image, "Press Q to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.imshow('Holistic Landmark Detection', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
len(pose_lm)

In [ ]:
# ── See ALL 33 landmark names with their index ────────────────────────────
print(f"Total landmarks detected: {len(pose_lm)}\n")
print("=" * 45)
print(f"{'INDEX':<8} {'NAME':<30} {'VISIBILITY'}")
print("=" * 45)

for lndmrk in mp_holistic.PoseLandmark:          # use mp_holistic, not mp_pose
    lm = pose_lm[lndmrk.value]                   # get the actual landmark data
    print(f"{lndmrk.value:<8} {lndmrk.name:<30} {lm.visibility:.2f}")

In [ ]:
pose_lm[mp_pose.PoseLandmark.LEFT_SHOULDER.value].visibility

In [ ]:
pose_lm[mp_pose.PoseLandmark.LEFT_ELBOW.value]

In [ ]:
pose_lm[mp_pose.PoseLandmark.LEFT_WRIST.value]

3. Calculate Angles

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Improved Angle Calculator
# ─────────────────────────────────────────────
def calculate_angle(a, b, c):
    """
    Calculate the angle at point B formed by segments A-B and C-B.
    Points are [x, y] normalized coordinates.
    Returns angle in degrees (0–180).
    """
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    c = np.array(c, dtype=float)

    ba = a - b   # vector from B to A
    bc = c - b   # vector from B to C

    # Dot product formula — more stable than arctan2 difference
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)   # guard against float errors
    angle = np.degrees(np.arccos(cosine_angle))

    return round(angle, 1) 

In [ ]:
def get_landmark(landmarks, name, side="LEFT"):
    """
    Safely extract [x, y] for a given landmark name and side.
    side: 'LEFT' or 'RIGHT'
    """
    full_name = f"{side}_{name}"
    lm = landmarks[mp_holistic.PoseLandmark[full_name].value]
    return [lm.x, lm.y]


def to_pixel(point, w=640, h=480):
    """Convert normalized [x,y] to pixel coordinates."""
    return tuple(np.multiply(point, [w, h]).astype(int))

In [ ]:
def draw_angle(image, angle, joint_pixel, good_range=(70, 160)):
    """
    Draw the angle value near the joint.
    Color: Green = good form, Yellow = warning, Red = bad form.
    """
    lo, hi = good_range
    if lo <= angle <= hi:
        color = (0, 255, 0)       # Green — good
    elif abs(angle - lo) < 15 or abs(angle - hi) < 15:
        color = (0, 255, 255)     # Yellow — borderline
    else:
        color = (0, 0, 255)       # Red — bad form

    cv2.putText(image, f"{angle}°",
                (joint_pixel[0] - 30, joint_pixel[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)

    # Draw a circle on the joint
    cv2.circle(image, joint_pixel, 6, color, -1)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

mp_drawing        = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic       = mp.solutions.holistic

# ─────────────────────────────────────────────
# Drawing Specs
# ─────────────────────────────────────────────
pose_landmark_spec   = mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2)
pose_connection_spec = mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
hand_landmark_spec   = mp_drawing.DrawingSpec(color=(0, 255, 0),    thickness=2, circle_radius=3)
hand_connection_spec = mp_drawing.DrawingSpec(color=(0, 128, 0),    thickness=2, circle_radius=2)
face_landmark_spec   = mp_drawing.DrawingSpec(color=(255, 224, 189),thickness=1, circle_radius=1)
face_connection_spec = mp_drawing.DrawingSpec(color=(200, 180, 150),thickness=1, circle_radius=1)

# ─────────────────────────────────────────────
# Angle Calculator
# ─────────────────────────────────────────────
def calculate_angle(a, b, c):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    c = np.array(c, dtype=float)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    return round(np.degrees(np.arccos(cosine_angle)), 1)

def get_landmark(landmarks, name, side="LEFT"):
    full_name = f"{side}_{name}"
    lm = landmarks[mp_holistic.PoseLandmark[full_name].value]
    return [lm.x, lm.y]

def to_pixel(point, w=640, h=480):
    return tuple(np.multiply(point, [w, h]).astype(int))

def draw_angle(image, angle, joint_pixel, good_range=(70, 160)):
    lo, hi = good_range
    if lo <= angle <= hi:
        color = (0, 255, 0)
    elif abs(angle - lo) < 15 or abs(angle - hi) < 15:
        color = (0, 255, 255)
    else:
        color = (0, 0, 255)
    cv2.putText(image, f"{angle}",
                (joint_pixel[0] - 30, joint_pixel[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
    cv2.circle(image, joint_pixel, 6, color, -1)

# ─────────────────────────────────────────────
# Rep Counter State
# ─────────────────────────────────────────────
counter = {"left": 0, "right": 0}
stage   = {"left": None, "right": None}

def count_reps(angle, side, up_thresh=160, down_thresh=90):
    if angle < down_thresh:
        stage[side] = "down"
    if angle > up_thresh and stage[side] == "down":
        stage[side] = "up"
        counter[side] += 1

# ─────────────────────────────────────────────
# Main Loop
# ─────────────────────────────────────────────
FRAME_W, FRAME_H = 640, 480
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  FRAME_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_H)

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    refine_face_landmarks=True,       # iris + precise eye landmarks
) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = holistic.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # ── 1. FACE MESH ──────────────────────────────────────────────────
        if results.face_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.face_landmarks,
                mp_holistic.FACEMESH_CONTOURS,
                landmark_drawing_spec=face_landmark_spec,
                connection_drawing_spec=face_connection_spec,
            )

        # ── 2. RIGHT HAND ─────────────────────────────────────────────────
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.right_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
                landmark_drawing_spec=hand_landmark_spec,
                connection_drawing_spec=hand_connection_spec,
            )

        # ── 3. LEFT HAND ──────────────────────────────────────────────────
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.left_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
                landmark_drawing_spec=hand_landmark_spec,
                connection_drawing_spec=hand_connection_spec,
            )

        # ── 4. FULL BODY POSE (includes legs!) ────────────────────────────
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_holistic.POSE_CONNECTIONS,
                landmark_drawing_spec=pose_landmark_spec,
                connection_drawing_spec=pose_connection_spec,
            )

        # ── 5. JOINT ANGLES — Arms + Hips + Knees + Ankles ───────────────
        try:
            lm = results.pose_landmarks.landmark

            for side in ["LEFT", "RIGHT"]:
                s = side.lower()

                # Grab all joints
                shoulder = get_landmark(lm, "SHOULDER", side)
                elbow    = get_landmark(lm, "ELBOW",    side)
                wrist    = get_landmark(lm, "WRIST",    side)
                hip      = get_landmark(lm, "HIP",      side)
                knee     = get_landmark(lm, "KNEE",     side)
                ankle    = get_landmark(lm, "ANKLE",    side)
                heel     = get_landmark(lm, "HEEL",     side)
                foot     = get_landmark(lm, "FOOT_INDEX", side)

                # Calculate all angles
                elbow_angle  = calculate_angle(shoulder, elbow,  wrist)
                hip_angle    = calculate_angle(shoulder, hip,    knee)
                knee_angle   = calculate_angle(hip,      knee,   ankle)
                ankle_angle  = calculate_angle(knee,     ankle,  heel)    # ← LEG angles

                # Rep counting via hip hinge
                count_reps(hip_angle, s)

                # Draw angles at each joint
                draw_angle(image, elbow_angle, to_pixel(elbow,  FRAME_W, FRAME_H), good_range=(30, 180))
                draw_angle(image, hip_angle,   to_pixel(hip,    FRAME_W, FRAME_H), good_range=(70, 180))
                draw_angle(image, knee_angle,  to_pixel(knee,   FRAME_W, FRAME_H), good_range=(90, 175))
                draw_angle(image, ankle_angle, to_pixel(ankle,  FRAME_W, FRAME_H), good_range=(70, 110))

            # ── Print key landmarks to console ────────────────────────────
            nose = lm[mp_holistic.PoseLandmark.NOSE.value]
            print(f"Nose → x:{nose.x:.2f}  y:{nose.y:.2f}  vis:{nose.visibility:.2f} | "
                  f"Reps L:{counter['left']}  R:{counter['right']}")

        except (AttributeError, KeyError):
            pass

        # ── 6. HUD Overlay ────────────────────────────────────────────────
        overlay = image.copy()
        cv2.rectangle(overlay, (0, 0), (280, 130), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.4, image, 0.6, 0, image)

        cv2.putText(image, "REPS",  (10, 25),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
        cv2.putText(image, "STAGE", (140, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)

        cv2.putText(image, f"L: {counter['left']}   R: {counter['right']}",
                    (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.putText(image, f"{stage['left']}  |  {stage['right']}",
                    (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2, cv2.LINE_AA)

        # Color legend at bottom
        cv2.putText(image, "GREEN=Good  YELLOW=Borderline  RED=Bad Form",
                    (10, FRAME_H - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)
        cv2.putText(image, "Q = Quit   R = Reset Reps",
                    (10, FRAME_H - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)

        cv2.imshow('Deadlift Tracker — Full Body', image)

        key = cv2.waitKey(10) & 0xFF
        if key == ord('q'):
            break
        if key == ord('r'):
            counter = {"left": 0, "right": 0}
            stage   = {"left": None, "right": None}

cap.release()
cv2.destroyAllWindows()

curl counter

In [ ]:
cap = cv2.VideoCapture(0)

# Curl counter variables
counter = 0 
stage = None

## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
      
        # Make detection
        results = pose.process(image)
    
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            
            # Get coordinates
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
            
            # Calculate angle
            angle = calculate_angle(shoulder, elbow, wrist)
            
            # Visualize angle
            cv2.putText(image, str(angle), 
                           tuple(np.multiply(elbow, [640, 480]).astype(int)), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA
                                )
            
            # Curl counter logic
            if angle > 160:
                stage = "down"
            if angle < 30 and stage =='down':
                stage="up"
                counter +=1
                print(counter)
                       
        except:
            pass
        
        # Render curl counter
        # Setup status box
        cv2.rectangle(image, (0,0), (225,73), (245,117,16), -1)
        
        # Rep data
        cv2.putText(image, 'REPS', (15,12), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), 
                    (10,60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA)
        
        # Stage data
        cv2.putText(image, 'STAGE', (65,12), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, stage, 
                    (60,60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA)
        
        
        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2), 
                                mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2) 
                                 )               
        
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

Final Code

In [34]:
import cv2
import mediapipe as mp
import numpy as np
from ultralytics import YOLO

# ─────────────────────────────────────────────
# CONFIG — Switch modes here
# ─────────────────────────────────────────────
SOURCE       = "videos/vid5.mp4"                    # 0 = live webcam | "videos/vid1.mp4" = video file
USE_YOLO     = True              # True = YOLO overlay | False = MediaPipe only
USE_MEDIAPIPE = False               # True = MediaPipe overlay | False = YOLO only
YOLO_MODEL   = "yolo26n-pose.pt"   # yolov8n-pose.pt / yolov8s-pose.pt / yolov8m-pose.pt
FRAME_W      = 720
FRAME_H      = 1020

# ─────────────────────────────────────────────
# Load YOLO Model
# ─────────────────────────────────────────────
yolo_model = YOLO(YOLO_MODEL) if USE_YOLO else None

# YOLO COCO keypoint order (17 points)
YOLO_KP = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# YOLO skeleton connections [from_idx, to_idx]
YOLO_SKELETON = [
    (0,1),(0,2),(1,3),(2,4),           # face
    (5,6),                              # shoulders
    (5,7),(7,9),                        # left arm
    (6,8),(8,10),                       # right arm
    (5,11),(6,12),(11,12),             # torso
    (11,13),(13,15),                    # left leg
    (12,14),(14,16),                    # right leg
]

# ─────────────────────────────────────────────
# MediaPipe Setup
# ─────────────────────────────────────────────
mp_drawing        = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic       = mp.solutions.holistic

pose_landmark_spec   = mp_drawing.DrawingSpec(color=(245,117,66),  thickness=2, circle_radius=2)
pose_connection_spec = mp_drawing.DrawingSpec(color=(245,66,230),  thickness=2, circle_radius=2)
hand_landmark_spec   = mp_drawing.DrawingSpec(color=(0,255,0),     thickness=2, circle_radius=3)
hand_connection_spec = mp_drawing.DrawingSpec(color=(0,128,0),     thickness=2, circle_radius=2)
face_landmark_spec   = mp_drawing.DrawingSpec(color=(255,224,189), thickness=1, circle_radius=1)
face_connection_spec = mp_drawing.DrawingSpec(color=(200,180,150), thickness=1, circle_radius=1)

# ─────────────────────────────────────────────
# Angle & Helper Functions
# ─────────────────────────────────────────────
def calculate_angle(a, b, c):
    a  = np.array(a, dtype=float)
    b  = np.array(b, dtype=float)
    c  = np.array(c, dtype=float)
    ba = a - b
    bc = c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return round(np.degrees(np.arccos(np.clip(cos, -1.0, 1.0))), 1)

def get_landmark(landmarks, name, side="LEFT"):
    lm = landmarks[mp_holistic.PoseLandmark[f"{side}_{name}"].value]
    return [lm.x, lm.y]

def to_pixel(point, w=FRAME_W, h=FRAME_H):
    return tuple(np.multiply(point, [w, h]).astype(int))

def draw_angle(image, angle, joint_pixel, good_range=(70, 160)):
    lo, hi = good_range
    if lo <= angle <= hi:
        color = (0, 255, 0)
    elif abs(angle - lo) < 15 or abs(angle - hi) < 15:
        color = (0, 255, 255)
    else:
        color = (0, 0, 255)
    cv2.putText(image, f"{angle}",
                (joint_pixel[0] - 30, joint_pixel[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
    cv2.circle(image, joint_pixel, 6, color, -1)

# ─────────────────────────────────────────────
# YOLO Drawing Function
# ─────────────────────────────────────────────
def draw_yolo_pose(image, yolo_results, conf_thresh=0.5):
    """Draw YOLO keypoints and skeleton on image."""
    for result in yolo_results:
        if result.keypoints is None:
            continue

        keypoints = result.keypoints.xy.cpu().numpy()    # [N_persons, 17, 2]
        confs     = result.keypoints.conf.cpu().numpy() if result.keypoints.conf is not None else None

        for person_idx, person_kps in enumerate(keypoints):
            # Draw skeleton connections
            for (i, j) in YOLO_SKELETON:
                if i >= len(person_kps) or j >= len(person_kps):
                    continue
                x1, y1 = int(person_kps[i][0]), int(person_kps[i][1])
                x2, y2 = int(person_kps[j][0]), int(person_kps[j][1])

                # Skip if coordinates are 0 (not detected)
                if x1 == 0 and y1 == 0: continue
                if x2 == 0 and y2 == 0: continue

                cv2.line(image, (x1, y1), (x2, y2), (0, 180, 255), 2, cv2.LINE_AA)

            # Draw keypoints
            for kp_idx, (x, y) in enumerate(person_kps):
                x, y = int(x), int(y)
                if x == 0 and y == 0:
                    continue

                conf = confs[person_idx][kp_idx] if confs is not None else 1.0
                if conf < conf_thresh:
                    continue

                # Color by body region
                if kp_idx < 5:
                    color = (255, 100, 100)   # Face — blue
                elif kp_idx < 11:
                    color = (100, 255, 100)   # Arms — green
                else:
                    color = (100, 100, 255)   # Legs — red

                cv2.circle(image, (x, y), 5, color, -1, cv2.LINE_AA)
                cv2.circle(image, (x, y), 7, (255, 255, 255), 1, cv2.LINE_AA)

            # Draw YOLO joint angles for first person only
            if person_idx == 0 and len(person_kps) >= 17:
                def kp(idx):
                    return [person_kps[idx][0], person_kps[idx][1]]

                def valid(idx):
                    return not (person_kps[idx][0] == 0 and person_kps[idx][1] == 0)

                # Left elbow
                if valid(5) and valid(7) and valid(9):
                    ang = calculate_angle(kp(5), kp(7), kp(9))
                    draw_angle(image, ang, (int(person_kps[7][0]), int(person_kps[7][1])), (30,180))

                # Right elbow
                if valid(6) and valid(8) and valid(10):
                    ang = calculate_angle(kp(6), kp(8), kp(10))
                    draw_angle(image, ang, (int(person_kps[8][0]), int(person_kps[8][1])), (30,180))

                # Left knee
                if valid(11) and valid(13) and valid(15):
                    ang = calculate_angle(kp(11), kp(13), kp(15))
                    draw_angle(image, ang, (int(person_kps[13][0]), int(person_kps[13][1])), (90,175))

                # Right knee
                if valid(12) and valid(14) and valid(16):
                    ang = calculate_angle(kp(12), kp(14), kp(16))
                    draw_angle(image, ang, (int(person_kps[14][0]), int(person_kps[14][1])), (90,175))

                # Left hip
                if valid(5) and valid(11) and valid(13):
                    ang = calculate_angle(kp(5), kp(11), kp(13))
                    draw_angle(image, ang, (int(person_kps[11][0]), int(person_kps[11][1])), (70,180))

                # Right hip
                if valid(6) and valid(12) and valid(14):
                    ang = calculate_angle(kp(6), kp(12), kp(14))
                    draw_angle(image, ang, (int(person_kps[12][0]), int(person_kps[12][1])), (70,180))

# ─────────────────────────────────────────────
# Rep Counter
# ─────────────────────────────────────────────
counter = {"left": 0, "right": 0}
stage   = {"left": None, "right": None}
frame_count = 0

def count_reps(angle, side, up_thresh=160, down_thresh=90):
    if angle < down_thresh:
        stage[side] = "down"
    if angle > up_thresh and stage[side] == "down":
        stage[side] = "up"
        counter[side] += 1

# ─────────────────────────────────────────────
# Open Video Source
# ─────────────────────────────────────────────
cap = cv2.VideoCapture(SOURCE)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  FRAME_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_H)

is_webcam = SOURCE == 0
print(f"▶ Source  : {'Webcam' if is_webcam else SOURCE}")
print(f"▶ YOLO    : {USE_YOLO}  |  MediaPipe: {USE_MEDIAPIPE}")
print(f"▶ Press Q to quit  |  R to reset reps  |  M toggle MediaPipe  |  Y toggle YOLO")

# ─────────────────────────────────────────────
# Main Loop
# ─────────────────────────────────────────────
with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    model_complexity=1,
    smooth_landmarks=True,
    refine_face_landmarks=True,
) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            # Loop video back to start when it ends
            if not is_webcam:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue
            break

        if is_webcam:
            frame = cv2.flip(frame, 1)

        frame_count += 1
        image = cv2.resize(frame, (FRAME_W, FRAME_H))

        # ── YOLO Inference ────────────────────────────────────────────────
        if USE_YOLO and yolo_model is not None:
            yolo_results = yolo_model.predict(
                image,
                conf=0.5,
                verbose=False,       # suppress per-frame console output
                stream=False,
            )
            draw_yolo_pose(image, yolo_results)

        # ── MediaPipe Inference ───────────────────────────────────────────
        if USE_MEDIAPIPE:
            rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results = holistic.process(rgb)
            rgb.flags.writeable = True

            # Face
            if results.face_landmarks:
                mp_drawing.draw_landmarks(
                    image, results.face_landmarks,
                    mp_holistic.FACEMESH_CONTOURS,
                    landmark_drawing_spec=face_landmark_spec,
                    connection_drawing_spec=face_connection_spec,
                )

            # Hands
            for hand_lm in [results.right_hand_landmarks, results.left_hand_landmarks]:
                if hand_lm:
                    mp_drawing.draw_landmarks(
                        image, hand_lm, mp_holistic.HAND_CONNECTIONS,
                        landmark_drawing_spec=hand_landmark_spec,
                        connection_drawing_spec=hand_connection_spec,
                    )

            # Pose skeleton
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    image, results.pose_landmarks,
                    mp_holistic.POSE_CONNECTIONS,
                    landmark_drawing_spec=pose_landmark_spec,
                    connection_drawing_spec=pose_connection_spec,
                )

            # Joint angles + rep counting
            try:
                lm = results.pose_landmarks.landmark

                for side in ["LEFT", "RIGHT"]:
                    s        = side.lower()
                    shoulder = get_landmark(lm, "SHOULDER", side)
                    elbow    = get_landmark(lm, "ELBOW",    side)
                    wrist    = get_landmark(lm, "WRIST",    side)
                    hip      = get_landmark(lm, "HIP",      side)
                    knee     = get_landmark(lm, "KNEE",     side)
                    ankle    = get_landmark(lm, "ANKLE",    side)
                    heel     = get_landmark(lm, "HEEL",     side)

                    elbow_angle = calculate_angle(shoulder, elbow, wrist)
                    hip_angle   = calculate_angle(shoulder, hip,   knee)
                    knee_angle  = calculate_angle(hip,      knee,  ankle)
                    ankle_angle = calculate_angle(knee,     ankle, heel)

                    count_reps(hip_angle, s)

                    draw_angle(image, elbow_angle, to_pixel(elbow),  (30,  180))
                    draw_angle(image, hip_angle,   to_pixel(hip),    (70,  180))
                    draw_angle(image, knee_angle,  to_pixel(knee),   (90,  175))
                    draw_angle(image, ankle_angle, to_pixel(ankle),  (70,  110))

                    # Live angle sidebar
                    debug_x = 10 if s == "left" else 340
                    cv2.putText(image, f"-- {side} --",      (debug_x, 155), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,0),   1)
                    cv2.putText(image, f"Elbow:{elbow_angle}",(debug_x, 172), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
                    cv2.putText(image, f"Hip  :{hip_angle}",  (debug_x, 189), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
                    cv2.putText(image, f"Knee :{knee_angle}", (debug_x, 206), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
                    cv2.putText(image, f"Ankle:{ankle_angle}",(debug_x, 223), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
                    cv2.putText(image, f"Stage:{stage[s]}",   (debug_x, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,255),   1)

            except (AttributeError, KeyError):
                pass

        # ── HUD ───────────────────────────────────────────────────────────
        overlay = image.copy()
        cv2.rectangle(overlay, (0, 0), (300, 140), (0,0,0), -1)
        cv2.addWeighted(overlay, 0.4, image, 0.6, 0, image)

        mode_label = "YOLO+MP" if (USE_YOLO and USE_MEDIAPIPE) else ("YOLO" if USE_YOLO else "MediaPipe")
        cv2.putText(image, f"Mode: {mode_label}",
                    (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,200,255), 1)
        cv2.putText(image, "REPS",  (10, 45),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
        cv2.putText(image, "STAGE", (150, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
        cv2.putText(image, f"L:{counter['left']}  R:{counter['right']}",
                    (10, 85), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0,255,0), 2, cv2.LINE_AA)
        cv2.putText(image, f"{stage['left']} | {stage['right']}",
                    (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,255,255), 2, cv2.LINE_AA)

        cv2.putText(image, "GREEN=Good  YELLOW=Warn  RED=Bad",
                    (10, FRAME_H - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200,200,200), 1)
        cv2.putText(image, "Q=Quit  R=Reset  M=Toggle MP  Y=Toggle YOLO",
                    (10, FRAME_H - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200,200,200), 1)

        src_label = "LIVE CAM" if is_webcam else f"VIDEO: {SOURCE}"
        cv2.putText(image, src_label, (FRAME_W - 180, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)

        cv2.imshow('Deadlift Tracker — YOLO + MediaPipe', image)

        # ── Key Controls ──────────────────────────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('r'):
            counter    = {"left": 0, "right": 0}
            stage      = {"left": None, "right": None}
        elif key == ord('y'):
            USE_YOLO = not USE_YOLO
            print(f"YOLO toggled → {USE_YOLO}")
        elif key == ord('m'):
            USE_MEDIAPIPE = not USE_MEDIAPIPE
            print(f"MediaPipe toggled → {USE_MEDIAPIPE}")

cap.release()
cv2.destroyAllWindows()

▶ Source  : videos/vid5.mp4
▶ YOLO    : True  |  MediaPipe: False
▶ Press Q to quit  |  R to reset reps  |  M toggle MediaPipe  |  Y toggle YOLO
